# 🔍 Project: RAG-Based FAQ Answering System
**NLP Course — Durham College HBAI Program**

A Retrieval-Augmented Generation (RAG) system that answers questions about the **Honours Bachelor of Artificial Intelligence (HBAI) program at Durham College** By: Khalfani Norman
1. Embedding a knowledge base of FAQ documents into a vector store (ChromaDB)
2. At query time, retrieving the top-3 most relevant chunks
3. Passing retrieved context + question to a local LLM (Flan-T5) to generate a grounded answer

**Stack:** `sentence-transformers` · `chromadb` · `transformers (flan-t5-base)` · No API key required

---

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q chromadb sentence-transformers transformers accelerate

## 📚 Step 2: Build the Knowledge Base

Curated FAQ corpus about the **Durham College HBAI program** — admission, curriculum, co-op, and graduation.

In [ ]:
faq_documents = [
    # Program Overview
    ("doc_001", "The Honours Bachelor of Artificial Intelligence (HBAI) is a four-year degree program "
               "offered at Durham College in Oshawa, Ontario. It is one of the first applied AI degrees "
               "in Canada and focuses on machine learning, deep learning, computer vision, NLP, and data engineering."),
    ("doc_002", "The HBAI program prepares graduates to design, build, and deploy AI systems. "
               "Students learn Python, TensorFlow, PyTorch, scikit-learn, OpenCV, Docker, SQL, "
               "and cloud platforms such as Google Cloud and AWS."),
    ("doc_003", "Durham College is located in Oshawa, Ontario, Canada, in the Durham Region east of Toronto. "
               "The campus is accessible by GO Transit and Durham Region Transit. "
               "The HBAI program is delivered primarily in-person with some hybrid components."),
    # Admission
    ("doc_004", "Admission to the HBAI program requires an OSSD with a minimum average of 70% in six "
               "Grade 12 U or M courses, including Grade 12 U English and Grade 12 U Mathematics "
               "(Advanced Functions or Calculus and Vectors)."),
    ("doc_005", "Mature students (21 years or older without an OSSD) may be considered for admission "
               "based on relevant work experience and academic upgrading. "
               "A personal statement and interview may be required."),
    ("doc_006", "International students must meet English language proficiency requirements: "
               "a minimum IELTS score of 6.5 overall (no band below 6.0), or equivalent TOEFL or Duolingo scores. "
               "Official transcripts must be translated into English if issued in another language."),
    # Program Structure
    ("doc_007", "The HBAI program is four years (eight semesters). Year 1 covers programming fundamentals, "
               "mathematics for AI, and introductory data science. Year 2 introduces machine learning, "
               "databases, and statistics. Year 3 focuses on deep learning, computer vision, NLP, and a co-op work term. "
               "Year 4 includes advanced AI topics and a capstone project."),
    ("doc_008", "Core courses include: Introduction to Programming (Python), Linear Algebra for AI, "
               "Probability and Statistics, Machine Learning, Deep Learning, Computer Vision, "
               "Natural Language Processing, Data Engineering, Cloud Computing for AI, Ethics in AI, and Capstone AI Project."),
    ("doc_009", "Students must complete a minimum of 130 credit hours across the four years. "
               "Each semester typically has five to six courses. "
               "Students must maintain a cumulative GPA of at least 2.0 to remain in good academic standing."),
    ("doc_010", "Electives include Reinforcement Learning, Generative AI, Robotics and Automation, "
               "Cybersecurity for AI Systems, Entrepreneurship in Tech, and Advanced Data Visualization. "
               "Students choose electives in Year 3 and Year 4."),
    # Co-op
    ("doc_011", "The HBAI program includes a mandatory co-op work term in the third year. "
               "Students complete one four-month paid work placement with an AI or technology company. "
               "Co-op placements are facilitated by Durham College Career and Employment Services and the DC Hired portal."),
    ("doc_012", "To be eligible for co-op, HBAI students must have completed all Year 1 and Year 2 courses "
               "with a minimum GPA of 2.5. Students must also complete co-op preparation workshops "
               "covering resume writing, interview skills, and professional networking."),
    ("doc_013", "Co-op employers include companies in the Greater Toronto Area and beyond, "
               "including AI startups, healthcare technology firms, financial services companies, "
               "and large technology corporations. Past hosts include companies in Oshawa, Toronto, Markham, and Mississauga."),
    ("doc_014", "During co-op, students apply AI skills such as data preprocessing, model training, "
               "deployment, or data analysis in a professional setting. "
               "A co-op reflection report must be submitted to Durham College at the end of the placement."),
    # Capstone
    ("doc_015", "The Capstone AI Project is a two-semester course in Year 4. Students work in small teams "
               "to design, build, and present an end-to-end AI solution for a real-world problem. "
               "Projects are presented to an industry panel at the end of Year 4."),
    ("doc_016", "Capstone projects have covered medical image analysis, predictive maintenance systems, "
               "NLP-based customer service bots, real-time object detection, and AI-powered financial forecasting. "
               "Projects must include a working prototype, GitHub repository, and written report."),
    # Graduation
    ("doc_017", "Graduates receive an Honours Bachelor of Artificial Intelligence degree from Durham College. "
               "This is a degree-level credential recognized by the Ontario government "
               "under the Post-secondary Education Choice and Excellence Act."),
    ("doc_018", "To graduate, students must complete all 130 required credit hours, maintain a cumulative GPA of 2.0+, "
               "successfully complete the co-op work term, and pass the Capstone AI Project. "
               "Convocation ceremonies are held in Spring and Fall."),
    # Tuition
    ("doc_019", "Domestic tuition is approximately $8,000 to $9,500 CAD per year for Ontario residents. "
               "International student tuition is approximately $16,000 to $19,000 CAD per year. "
               "Fees vary by year and are subject to annual adjustments."),
    ("doc_020", "Financial aid options include OSAP (Ontario Student Assistance Program), "
               "Durham College bursaries and scholarships, the Ontario Learn and Stay Grant, "
               "and entrance awards for high-achieving incoming students. Apply for OSAP before each academic year."),
    # Resources
    ("doc_021", "HBAI students have access to AI and Data Analytics labs with high-performance GPU workstations, "
               "cloud computing credits (Google Cloud and AWS), and licensed software "
               "including MATLAB, Tableau, and JetBrains IDEs."),
    ("doc_022", "Durham College offers academic support: tutoring, the Academic Learning Services (ALS) centre, "
               "mental health and wellness counselling, library and research databases, "
               "and accessibility services for students with disabilities."),
    ("doc_023", "Student clubs include the DC AI Club, Coding and Robotics Club, and the Entrepreneurship Hub. "
               "Students participate in intercollegiate hackathons and data science competitions "
               "such as Kaggle and MLH events."),
    # Careers
    ("doc_024", "Common job titles for HBAI graduates include Machine Learning Engineer, Data Scientist, "
               "AI Developer, Computer Vision Engineer, NLP Engineer, MLOps Engineer, and AI Research Analyst. "
               "Graduates work in healthcare, finance, retail, manufacturing, and government."),
    ("doc_025", "HBAI graduates may pursue a Masters degree in Computer Science, Artificial Intelligence, "
               "Data Science, or related fields at University of Toronto, York University, "
               "Ontario Tech University, or McMaster University."),
    # Application & Contact
    ("doc_026", "Applications are submitted through OCAS at ontariocolleges.ca. "
               "The application deadline for the September intake is typically February 1st. "
               "A limited number of spots may be available for late applicants."),
    ("doc_027", "For information about HBAI, contact Durham College Admissions at admissions@durhamcollege.ca "
               "or call 905-721-2000. Program-specific inquiries go to the School of Artificial Intelligence faculty office."),
]

print(f"Knowledge base loaded: {len(faq_documents)} documents")
for doc_id, text in faq_documents[:3]:
    print(f"[{doc_id}] {text[:100]}...")

## 🔢 Step 3: Load Embedding Model

Using **`all-MiniLM-L6-v2`** — fast, lightweight, runs on CPU.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded: all-MiniLM-L6-v2")

test_emb = embedding_model.encode("What is the HBAI program?")
print(f"Embedding dimension: {len(test_emb)}")

## 🗄️ Step 4: Store Embeddings in ChromaDB

Embed each FAQ document and store in an in-memory ChromaDB vector collection.

In [ ]:
import chromadb

chroma_client = chromadb.Client()

collection_name = "hbai_faq"
try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

collection = chroma_client.create_collection(name=collection_name)

doc_ids   = [doc[0] for doc in faq_documents]
doc_texts = [doc[1] for doc in faq_documents]

print("Embedding documents...")
doc_embeddings = embedding_model.encode(doc_texts, show_progress_bar=True).tolist()

collection.add(
    documents=doc_texts,
    embeddings=doc_embeddings,
    ids=doc_ids
)

print(f"\n{collection.count()} documents stored in ChromaDB collection '{collection_name}'")

## 🤖 Step 5: Load Local LLM (Flan-T5-Base)

Using **`google/flan-t5-base`** — free, instruction-tuned, runs on CPU. No API key needed.

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

model_name = "google/flan-t5-base"
print(f"Loading LLM: {model_name}...")

tokenizer  = T5Tokenizer.from_pretrained(model_name)
llm_model  = T5ForConditionalGeneration.from_pretrained(model_name)
llm_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

print(f"LLM loaded on: {device}")

## 🔧 Step 6: Define the RAG Pipeline

Three stages:
1. **Retrieve** — embed the question, fetch top-3 chunks from ChromaDB
2. **Augment** — build a grounded prompt with context + question
3. **Generate** — pass to Flan-T5, return answer with source references

In [ ]:
def retrieve_chunks(question, top_k=3):
    """Embed question and retrieve top-k relevant document chunks."""
    query_embedding = embedding_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    chunks = []
    for doc_id, doc_text, distance in zip(
        results["ids"][0], results["documents"][0], results["distances"][0]
    ):
        chunks.append({"id": doc_id, "text": doc_text, "similarity": round(1 - distance, 4)})
    return chunks


def build_prompt(question, chunks):
    """Build RAG prompt with retrieved context chunks."""
    context = "\n\n".join([f"[{c['id']}] {c['text']}" for c in chunks])
    return (
        "Answer the following question using ONLY the context provided below. "
        "If the answer is not in the context, say 'I don't have enough information to answer that.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer:"
    )


def generate_answer(prompt, max_new_tokens=200):
    """Pass prompt to Flan-T5 and return the generated answer."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = llm_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def rag_answer(question, top_k=3, verbose=True):
    """Full RAG pipeline: retrieve -> augment -> generate."""
    chunks = retrieve_chunks(question, top_k=top_k)
    prompt = build_prompt(question, chunks)
    answer = generate_answer(prompt)

    if verbose:
        print("=" * 60)
        print(f"Question: {question}")
        print("=" * 60)
        print(f"Answer:\n{answer}")
        print("\nSource Chunks Retrieved:")
        for c in chunks:
            print(f"  [{c['id']}] similarity={c['similarity']} — {c['text'][:100]}...")
        print("=" * 60)

    return {"question": question, "answer": answer,
            "sources": [c["id"] for c in chunks], "chunks": chunks}


print("RAG pipeline functions defined.")

## 🧪 Step 7: Demo — Ask Questions

In [ ]:
result = rag_answer("What are the admission requirements for the HBAI program?")

In [ ]:
result = rag_answer("How does the co-op work term work?")

In [ ]:
result = rag_answer("What programming languages and tools do HBAI students learn?")

In [ ]:
result = rag_answer("How much does tuition cost for international students?")

In [ ]:
result = rag_answer("What is the best restaurant near campus?")  # Out-of-scope test

## 📊 Step 8: Evaluation — Retrieval Recall@3 & Answer Faithfulness

Evaluated on **30 test questions** using two metrics:
- **Retrieval Recall@3**: Did the correct document appear in the top-3 results?
- **Answer Faithfulness**: Does the answer contain keywords from the expected answer? (proxy metric)

In [ ]:
test_questions = [
    # Admission (5)
    ("What GPA or grades are needed to apply to HBAI?",              "doc_004", ["70%", "Grade 12", "OSSD"]),
    ("Can mature students apply to the HBAI program?",               "doc_005", ["mature", "21", "experience"]),
    ("What English test scores do international applicants need?",   "doc_006", ["IELTS", "6.5", "English"]),
    ("How do I apply to Durham College HBAI?",                       "doc_026", ["OCAS", "ontariocolleges", "February"]),
    ("What is the application deadline for HBAI?",                   "doc_026", ["February", "deadline", "September"]),
    # Program overview (5)
    ("What is the HBAI program?",                                    "doc_001", ["Artificial Intelligence", "Durham College", "degree"]),
    ("Where is Durham College located?",                             "doc_003", ["Oshawa", "Ontario", "Toronto"]),
    ("What tools and languages do students learn in HBAI?",          "doc_002", ["Python", "TensorFlow", "PyTorch"]),
    ("How many years is the HBAI program?",                          "doc_007", ["four", "year", "semester"]),
    ("What courses are in the HBAI curriculum?",                     "doc_008", ["Machine Learning", "Computer Vision", "NLP"]),
    # Co-op (5)
    ("Is co-op mandatory in HBAI?",                                  "doc_011", ["mandatory", "co-op", "work term"]),
    ("When does the co-op happen in the HBAI program?",              "doc_011", ["third year", "four-month", "placement"]),
    ("What GPA do I need for the co-op term?",                       "doc_012", ["2.5", "GPA", "eligible"]),
    ("Where do HBAI students do their co-op?",                       "doc_013", ["Toronto", "GTA", "companies"]),
    ("What do students do during co-op?",                            "doc_014", ["AI skills", "model", "report"]),
    # Capstone (2)
    ("What is the capstone project in HBAI?",                        "doc_015", ["Year 4", "team", "AI solution"]),
    ("What have past capstone projects covered?",                    "doc_016", ["medical", "NLP", "prototype"]),
    # Graduation (3)
    ("What credential do HBAI graduates receive?",                   "doc_017", ["Honours Bachelor", "degree", "Ontario"]),
    ("What are the graduation requirements for HBAI?",               "doc_018", ["130", "GPA", "co-op", "Capstone"]),
    ("How many credit hours are needed to graduate?",                "doc_009", ["130", "credit hours", "GPA"]),
    # Tuition (3)
    ("How much is tuition for domestic students?",                   "doc_019", ["8,000", "9,500", "CAD"]),
    ("What is the international tuition for HBAI?",                  "doc_019", ["16,000", "19,000", "international"]),
    ("What financial aid is available for HBAI students?",           "doc_020", ["OSAP", "bursaries", "scholarship"]),
    # Resources (3)
    ("What labs and equipment do HBAI students have access to?",     "doc_021", ["GPU", "Google Cloud", "AWS"]),
    ("What academic support does Durham College offer?",             "doc_022", ["tutoring", "ALS", "counselling"]),
    ("Are there student clubs for HBAI students?",                   "doc_023", ["AI Club", "hackathon", "Kaggle"]),
    # Careers (2)
    ("What jobs can HBAI graduates get?",                            "doc_024", ["Machine Learning", "Data Scientist", "NLP"]),
    ("Can HBAI graduates go to grad school?",                        "doc_025", ["Master", "University of Toronto", "Data Science"]),
    # Electives & Contact (2)
    ("What elective courses are available in HBAI?",                 "doc_010", ["Reinforcement Learning", "Generative AI", "Robotics"]),
    ("How can I contact Durham College about HBAI admissions?",      "doc_027", ["admissions", "durhamcollege.ca", "905"]),
]

print(f"Test set ready: {len(test_questions)} questions")

In [ ]:
import csv

results_log = []
recall_hits = 0
faithfulness_hits = 0

print("Running evaluation on 30 test questions...\n")

for i, (question, expected_doc, keywords) in enumerate(test_questions):
    result = rag_answer(question, top_k=3, verbose=False)

    # Metric 1: Retrieval Recall@3
    recall_hit = expected_doc in result["sources"]
    if recall_hit:
        recall_hits += 1

    # Metric 2: Answer Faithfulness (keyword proxy)
    answer_lower = result["answer"].lower()
    matched_keywords = [kw for kw in keywords if kw.lower() in answer_lower]
    faithful = len(matched_keywords) >= 1
    if faithful:
        faithfulness_hits += 1

    results_log.append({
        "q_num": i + 1, "question": question, "expected_doc": expected_doc,
        "retrieved_docs": ", ".join(result["sources"]), "recall_hit": recall_hit,
        "answer": result["answer"], "keywords_matched": ", ".join(matched_keywords), "faithful": faithful,
    })

    status = "PASS" if (recall_hit and faithful) else ("PARTIAL" if recall_hit else "FAIL")
    print(f"[{status}] Q{i+1:02d}: {question[:65]}")

total = len(test_questions)
print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"Retrieval Recall@3:   {recall_hits}/{total} = {recall_hits/total:.1%}")
print(f"Answer Faithfulness:  {faithfulness_hits}/{total} = {faithfulness_hits/total:.1%}")
print("=" * 50)

In [ ]:
fieldnames = ["q_num", "question", "expected_doc", "retrieved_docs",
              "recall_hit", "answer", "keywords_matched", "faithful"]

with open("test_questions.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results_log)

print("Results saved to test_questions.csv")

## 📈 Step 9: Failure Analysis

In [ ]:
failures = [r for r in results_log if not r["recall_hit"] or not r["faithful"]]
print(f"Failure Cases: {len(failures)} / {len(test_questions)}\n")

for r in failures:
    print(f"Q{r['q_num']:02d}: {r['question']}")
    print(f"  Expected doc : {r['expected_doc']}")
    print(f"  Retrieved    : {r['retrieved_docs']}")
    print(f"  Recall hit   : {r['recall_hit']}")
    print(f"  Answer       : {r['answer'][:150]}")
    print(f"  Faithful     : {r['faithful']}")
    print()

## 💬 Step 10: Interactive Demo — Ask Your Own Question

In [ ]:
your_question = "What can I do after graduating from the HBAI program?"  # Change this!
result = rag_answer(your_question)

---

## 📝 Evaluation Report Summary

### System Overview
RAG system answering questions about the Durham College HBAI program using a 27-document knowledge base. Embeds with `all-MiniLM-L6-v2` (384-dim), stores in ChromaDB, generates with `google/flan-t5-base`.

### Metrics
| Metric | Description | Target |
|---|---|---|
| **Retrieval Recall@3** | Was the correct doc in the top-3 results? | ≥ 80% |
| **Answer Faithfulness** | Does the answer include expected keywords? | ≥ 70% |

### Known Limitations
- Flan-T5-base is small — occasionally truncates answers awkwardly. Upgrade to `flan-t5-large` for better quality.
- Faithfulness metric is a keyword proxy — a full faithfulness score would use an NLI model.
- Knowledge base is static — cannot answer questions about events after the last KB update.

### Possible Improvements
- Chunk long documents into smaller overlapping segments for finer-grained retrieval
- Add re-ranking with a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`)
- Replace Flan-T5 with a larger open-source LLM (e.g., Mistral-7B via Ollama)
- Add a Gradio UI for interactive demos